# Case-Based Reasoning in Machine Learning - Starter Notebook
Case-Based Reasoning (CBR) solves new problems by retrieving and adapting similar past cases.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Past medical-like cases (toy example)
cases = pd.DataFrame({
    'fever': [1, 1, 0, 0, 1, 0, 1, 0],
    'cough': [1, 0, 1, 0, 1, 0, 0, 1],
    'fatigue': [1, 1, 0, 0, 1, 0, 1, 0],
    'age': [67, 51, 28, 35, 73, 45, 61, 32],
    'risk_label': ['high', 'medium', 'low', 'low', 'high', 'medium', 'high', 'low']
})
print(cases)

In [ ]:
# New query case
query = pd.DataFrame({'fever': [1], 'cough': [1], 'fatigue': [1], 'age': [64]})

X = cases[['fever', 'cough', 'fatigue', 'age']].values
Xq = query.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
Xq_scaled = scaler.transform(Xq)

sim = cosine_similarity(X_scaled, Xq_scaled).ravel()
cases['similarity'] = sim
top_k = cases.sort_values('similarity', ascending=False).head(3)
print('Top-3 retrieved cases:')
print(top_k[['fever', 'cough', 'fatigue', 'age', 'risk_label', 'similarity']])

In [ ]:
# Reuse/adapt step: weighted vote by similarity
label_to_score = {'low': 0, 'medium': 1, 'high': 2}
score_to_label = {0: 'low', 1: 'medium', 2: 'high'}

weighted_sum = 0.0
weight_total = 0.0
for _, row in top_k.iterrows():
    w = max(row['similarity'], 0)
    weighted_sum += w * label_to_score[row['risk_label']]
    weight_total += w

pred_score = weighted_sum / weight_total if weight_total > 0 else 1
pred_label = score_to_label[int(round(pred_score))]
print(f'Predicted query risk by CBR: {pred_label} (score={pred_score:.2f})')

print('\nCBR cycle recap: Retrieve -> Reuse -> (Revise) -> Retain')